# 영화 리뷰 워드 임베딩 (Word2Vec, FastText)
- gensim 라이브러리 사용 : pip install gensim
    - Word2Vec : models.Word2Vec
    - FastText : models.FastText

## 1. 데이터 준비
* 토큰화가 잘 되어 있는 filtered 데이터 사용

In [1]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016_filtered.csv'
data_df = pd.read_csv(data_filename)
data_df.head()


,review,rate
0,아니 딴 그렇 비 비탄 총 대체 왜 들 온겨,7
1,진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임,1
2,역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 ...,10
3,온종일 불편한 피 범벅 일,6
4,답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐,1


In [2]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 788189 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    788189 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.0+ MB


In [3]:
# 결측치 제거 (rate에 결측치)
# 결측치를 제거하지 않으면 review 데이터 추출 시 float으로 타입 변환됨
data_df.dropna(inplace=True)

In [4]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 785448 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    785448 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 18.0+ MB


In [5]:
# review만 모아서 review별 토큰 리스트로 변환
review_list = list(map(str,data_df.review))
review_list[:5]



['아니 딴 그렇 비 비탄 총 대체 왜 들 온겨',
 '진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임',
 '역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 실사 했 사람 거르려 그냥 봤 정말 흠잡 없는 최고 좀비 영화 잔인 거 싫어하지 참고 볼 만하 로미 인물 왜 그런 모르',
 '온종일 불편한 피 범벅 일',
 '답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐']

In [6]:
type(review_list[0])

str

In [8]:
# 결측치 처리 시 AttributeError 예외 발생
token_list = [review.split() for review in review_list]
print(token_list[:2])

[['아니', '딴', '그렇', '비', '비탄', '총', '대체', '왜', '들', '온겨'], ['진심', '쓰레기', '영화', '만들', '무서', '알', '쫄아', '틀었', '이건', '뭐', '웃', '거리', '없는', '쓰레기', '영화', '임']]


## 1. Word2Vec 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/word2vec.html

### Skipgram, negative=10 인 경우

In [9]:
# Word2Vec 모델 생성 및 학습 : window=3, min_count=3
from gensim.models import Word2Vec
model_sg_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=10)


In [10]:
# 단어의 임베딩 벡터 확인
model_sg_n10.wv['이정재']

array([ 0.02332699,  0.09651224, -0.02136111, -0.4781899 ,  0.00110965,
       -0.06148559, -0.14274076, -0.04481744,  1.0319237 , -0.13395368,
        0.17429954, -0.43030486,  0.04469056,  0.42727768, -0.06051938,
       -0.19309531, -0.16307272,  0.1572356 ,  0.42316142, -0.27228296,
        0.15452273,  0.21400683,  0.12537451,  0.10892601, -0.30560702,
       -0.3606539 ,  0.22669616, -0.14529637, -0.19131096, -0.17413871,
        0.2713924 , -0.11294182,  0.05550312,  0.02365061, -0.22050169,
       -0.21589485, -0.36243033,  0.04566554, -0.5246605 , -0.27842283,
       -0.21188562,  0.15522625, -0.29473692, -0.10565994,  0.00813763,
        0.13486661,  0.18608215, -0.27137282, -0.45579448, -0.01434302,
        0.35181606, -0.42585048, -0.21536586, -0.09643929,  0.21359566,
        0.03444755,  0.25367123, -0.06444554,  0.04625055,  0.15444207,
       -0.41028777,  0.12059429,  0.60644585, -0.24886113, -0.57394946,
       -0.14572243, -0.20822504, -0.43318644,  0.3327289 , -0.22

In [11]:
# 단어의 임베딩 벡터 차원 확인
len(model_sg_n10.wv['이정재'])


100

In [12]:
# 두 단어 간 유사도 확인
wv = model_sg_n10.wv
wv.similarity('이정재', '정우성')


np.float32(0.7805648)

In [13]:
# 특정 단어와 유사한 단어 추출
wv.most_similar('이정재', topn=20)


[('이범수', 0.8283710479736328),
 ('공유', 0.8175774216651917),
 ('송강호', 0.7990674376487732),
 ('김범수', 0.7849009037017822),
 ('정우성', 0.7805647850036621),
 ('조재현', 0.7756615877151489),
 ('김남길', 0.7413871884346008),
 ('박해일', 0.737578272819519),
 ('주지훈', 0.7353487610816956),
 ('이성민', 0.734247624874115),
 ('이병헌', 0.7276780009269714),
 ('이재훈', 0.7208290100097656),
 ('곽도원', 0.7205561399459839),
 ('이진욱', 0.7201042771339417),
 ('김성균', 0.7151978015899658),
 ('김윤석', 0.7140246033668518),
 ('마동석', 0.7133253812789917),
 ('요한', 0.7120636701583862),
 ('김명민', 0.7120574712753296),
 ('송광호', 0.7071585655212402)]

In [14]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '잼남', '재밌었', '재밌어', '재밋음', '재밌슴', '잼슴', '재밋구', '재밋엇음', '재밋엇어용', '재밋어용', '재미있었', '재밌아', '재밋엇', '쟈밋', '재밋었어', '재미있더', '재밋네', '존잼임', '재밋게봣', '엇', '재밋엇어', '재밌던', '재밋습니', '재미있네', '재밋었음', '재밋게봣어', '재밋는듯', '재밌고', '잼잇엇음', '재밌드', '꿀잼임', '재밌엇습니', '번보', '재밋네용', '재밋습니당', '재미있구', '재밌더', '강추함', '재미나다', '재밋어', '재밋게봄', '짱재밋어', '재밌는', '재밋당', '재밋습', '재밋었습니', '재밋었네', '재밋엇습니']


### Skipgram, negative=5 인 경우

In [15]:
# 모델 생성
model_sg_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=5)


In [16]:
# 특어 단어와 유사한 단어 추출 : 이정재
wv = model_sg_n5.wv
wv.most_similar('이정재', topn=20)



[('이범수', 0.8213378190994263),
 ('공유', 0.7809830904006958),
 ('송강호', 0.7786644101142883),
 ('김범수', 0.7500667572021484),
 ('이병헌', 0.742543637752533),
 ('김남길', 0.7231557369232178),
 ('정우성', 0.712207555770874),
 ('김명민', 0.7092728018760681),
 ('곽도원', 0.7084077000617981),
 ('리암', 0.7030840516090393),
 ('주지훈', 0.7025614976882935),
 ('박해일', 0.6955569982528687),
 ('슨', 0.6947159171104431),
 ('마동석', 0.6907256245613098),
 ('송광호', 0.6886768341064453),
 ('조재현', 0.6884209513664246),
 ('이성민', 0.6832793951034546),
 ('조진웅', 0.6825048327445984),
 ('이진욱', 0.681090235710144),
 ('배성우', 0.6768836379051208)]

In [17]:
# 특어 단어와 유사한 단어 추출 : 재밌
print([word for word, _ in wv.most_similar('재밌', topn=50)])


['재미있', '재밌네', '재밌었', '잼남', '재밋음', '재밌어', '재밋엇음', '잼슴', '재밋었음', '재밋엇', '재밋었어', '재밋엇어용', '재밋어용', '재밌아', '재밋네용', '재밋습니', '재밋네', '재미있었', '재밋구', '쟈밋', '재밋엇어', '재밋어', '재밋게봣어', '재밌슴', '엇', '존잼임', '재밋게봣습니', '재밋는듯', '꿀잼', '재밋었습니', '꿀잼임', '재미있더', '개꿀잼', '재밋엇습니', '짱잼', '재밋게봣', '재미나다', '꾸르잼', '재미있네', '재밌던', '재밋게봣음', '존잼', '재밋습니당', '재밋습', '재밋당', '재밋게봄', '재밌더', '재밌엇습니', '핵꿀잼', '재밋었']


### CBOW, negative=10 인 경우

In [18]:
model_cbow_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=10)

In [19]:
wv = model_cbow_n10.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.7878981232643127),
 ('공유', 0.7611801028251648),
 ('김윤석', 0.7563378810882568),
 ('주지훈', 0.7353175282478333),
 ('조재현', 0.7315343022346497),
 ('김범수', 0.7301983833312988),
 ('이성민', 0.7237082123756409),
 ('송강호', 0.715438723564148),
 ('박해일', 0.7065120339393616),
 ('김남길', 0.7057331204414368),
 ('이진욱', 0.6987204551696777),
 ('민호', 0.6773200035095215),
 ('차승원', 0.6770884990692139),
 ('김성균', 0.6743884086608887),
 ('마동석', 0.6720434427261353),
 ('정우성', 0.6706122756004333),
 ('이희준', 0.6695676445960999),
 ('조정석', 0.6684679985046387),
 ('김명민', 0.668261706829071),
 ('강예원', 0.6662810444831848)]

In [20]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌어', '재밋음', '재밌었', '잼남', '재밋어', '재밌는', '재미있었', '재밌더', '재밌던', '재미있네', '재미있어', '재밋엇어', '재밋', '재밋네', '꿀잼', '재밋엇', '재밋었어', '재미있던', '재밌고', '재밋었', '재밌다', '재밋엇음', '재미있는', '재밌게', '재밌구', '잼', '재밋었음', '재밋습니', '재미있더', '웃김', '개꿀잼', '재미있다', '존잼', '재미있고', '재미있게', '무서웠', '재밌으', '괜찮', '재미나', '웃겼', '멋있', '재미없', '재미있겠', '재밋어용', '재미있구', '무서', '핵꿀잼', '재밌겠']


### CBOW, negative=5 인 경우

In [21]:
model_cbow_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=5)

In [22]:
wv = model_cbow_n5.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.7872775197029114),
 ('공유', 0.7837879657745361),
 ('송강호', 0.7303721904754639),
 ('김윤석', 0.728175938129425),
 ('이성민', 0.7174971103668213),
 ('조재현', 0.7090469598770142),
 ('김남길', 0.6917862892150879),
 ('이진욱', 0.6898272037506104),
 ('주지훈', 0.6893651485443115),
 ('김범수', 0.6699015498161316),
 ('박해일', 0.6683788895606995),
 ('조진웅', 0.6520740985870361),
 ('곽도원', 0.6485368013381958),
 ('조정석', 0.6482192873954773),
 ('하정우', 0.6480566263198853),
 ('차승원', 0.6470074653625488),
 ('김성오', 0.6464999914169312),
 ('정우성', 0.6434531211853027),
 ('민호', 0.6409621238708496),
 ('공효진', 0.6369225978851318)]

In [23]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밋음', '재밌었', '재밌어', '재밋어', '재미있었', '재밌는', '재미있네', '재밌더', '잼남', '재밌던', '재밋네', '재밋엇어', '재밋엇', '재미있어', '재밌다', '재밋', '꿀잼', '재밌고', '재밋었어', '재미있는', '재미있던', '재밋엇음', '재밋었', '재미있더', '재밌게', '재밌구', '재밋었음', '잼', '재미있다', '웃김', '개꿀잼', '재밋습니', '재밌으', '재미있고', '재미있게', '재미있겠', '재밌겠', '무서웠', '존잼', '재미없', '재밋었습니', '괜찮', '재미나', '웃겼', '멋있', '졸잼', '재미있구', '재밋어용']


### OOV(Out of Vocabulary) 문제

In [24]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in model_sg_n10.wv.key_to_index

False

In [26]:
# corpus에 없는 단어의 임베딩 벡터 확인 
model_sg_n10.wv['우주평화']

KeyError: "Key '우주평화' not present"

## 2. FastText 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/fasttext.html

In [27]:
# FastText 모델 생성 및 학습
# window=3, min_count=3, min_n=2, max_n=2
from gensim.models import FastText
model = FastText(token_list, vector_size=100, window=3, min_count=3,sg=1, negative=10, min_n=2, max_n=2)
wv = model.wv

In [28]:
# 특정 단어와 유사한 단어 추출 : 이정재
wv.most_similar('이정재', topn=20)

[('이범수', 0.8391290903091431),
 ('정재영', 0.8291296362876892),
 ('정재', 0.8225804567337036),
 ('정재형', 0.8046644330024719),
 ('공유', 0.8023857474327087),
 ('송강호', 0.802384078502655),
 ('김범수', 0.7865857481956482),
 ('요한', 0.7770371437072754),
 ('김철민', 0.7648900151252747),
 ('이정진', 0.7643253803253174),
 ('김지민', 0.7612342834472656),
 ('이정현', 0.7593178153038025),
 ('임성민', 0.7586401700973511),
 ('김성균', 0.757475733757019),
 ('톰슨', 0.7572811245918274),
 ('이진욱', 0.7555825710296631),
 ('박해일', 0.7527135610580444),
 ('조재현', 0.7521123886108398),
 ('정우성', 0.75071781873703),
 ('이재훈', 0.7471041679382324)]

In [29]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in wv.key_to_index 

False

In [30]:
# corpus에 없는 단어의 임베딩 벡터 확인 
wv['우주평화']


array([ 2.15281963e-01,  2.10137106e-02,  1.26944572e-01,  9.91101414e-02,
       -5.32898586e-03, -1.96972772e-01, -4.11075652e-01,  8.35509956e-01,
        5.54459155e-01,  1.86358780e-01, -1.41220167e-01,  1.44580111e-01,
        1.98259041e-01,  3.59125614e-01, -2.76112944e-01, -3.86854321e-01,
        4.64522727e-02, -1.69645712e-01,  1.75410345e-01,  2.64587283e-01,
        1.83349505e-01,  4.04838333e-03,  1.09360732e-01,  4.01763208e-02,
        1.76458567e-01,  5.49316392e-05, -4.48208958e-01, -3.18679422e-01,
       -2.46852040e-02, -3.01972330e-01,  1.68445215e-01, -4.06745851e-01,
        2.55188286e-01, -4.39864933e-01, -1.70556903e-02,  3.60923320e-01,
        6.41709268e-02,  1.25575349e-01, -2.39663452e-01, -7.25498563e-03,
       -2.50164211e-01, -1.31190211e-01, -2.55329907e-01, -4.42089140e-03,
       -1.01101641e-02,  1.78876996e-01, -1.16627455e-01,  8.82005244e-02,
       -1.18872270e-01,  1.29801244e-01,  3.58828843e-01,  7.89100081e-02,
       -1.56972140e-01, -

In [31]:
# corpus에 없는 단어와 유사한 단어추출 
wv.most_similar('우주평화', topn=20)


[('우주', 0.8375473022460938),
 ('평화', 0.8143447637557983),
 ('우주비행사', 0.8031156063079834),
 ('우주여행', 0.789772629737854),
 ('투명인간', 0.788837194442749),
 ('우주선', 0.7879127264022827),
 ('우장', 0.7830451130867004),
 ('우방', 0.7830129861831665),
 ('우주인', 0.7810357809066772),
 ('지구인', 0.7792947888374329),
 ('산화', 0.7792237401008606),
 ('회색곰', 0.7788624167442322),
 ('장래희망', 0.7777155041694641),
 ('켤', 0.7762529850006104),
 ('지구대', 0.7755840420722961),
 ('지구촌', 0.7753890156745911),
 ('고물상', 0.774395763874054),
 ('볐', 0.7731928825378418),
 ('봉하마을', 0.7711992263793945),
 ('상궁', 0.7701133489608765)]